In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04. Full Pipeline Demo\n",
    "End-to-end Monte Carlo simulation. Uses the GAN only if a trained checkpoint is supplied (see notebook 02), otherwise correlated GBM with jumps. The LLM step is optional."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "import json\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.pipeline import MonteCarloPipeline\n",
    "\n",
    "%matplotlib inline\n",
    "print('[INFO] Environment ready')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "tickers = ['AAPL', 'MSFT', 'GOOGL']\n",
    "\n",
    "pipeline = MonteCarloPipeline(n_assets=len(tickers), n_simulations=5000, filter_top_k=0.01,\n",
    "                              use_gan=False, use_live_data=False)\n",
    "# To use a trained GAN:\n",
    "# pipeline.use_gan = True\n",
    "# pipeline.load_models(checkpoint_path='../data/models/dfmgan.pth')\n",
    "print('[SUCCESS] Pipeline ready | GAN:', pipeline.use_gan)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "results = pipeline.run_simulation(tickers, period='2y')\n",
    "meta = results['metadata']\n",
    "used = meta['tickers']\n",
    "\n",
    "print('Simulation completed in', round(meta['computation_time'], 2), 's')\n",
    "print('Paths:', meta['n_simulations'], '| display sample kept:', meta['filtered_paths'])\n",
    "print('Data source:', meta['data_source'])\n",
    "if meta['missing_tickers']:\n",
    "    print('[WARNING] No data for:', meta['missing_tickers'])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "expected_prices = results['expected_prices']\n",
    "confidence_intervals = results['confidence_intervals']\n",
    "\n",
    "print('Expected prices (end of horizon):')\n",
    "for t in used:\n",
    "    exp = expected_prices[t]\n",
    "    lo, hi = confidence_intervals[t]\n",
    "    print(f'  {t}: {exp:.2f}  [{lo:.2f}, {hi:.2f}]')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "risk_metrics = results['risk_metrics']\n",
    "\n",
    "for t, m in risk_metrics.items():\n",
    "    var95 = m.get('var_95', 0) * 100\n",
    "    cvar95 = m.get('cvar_95', 0) * 100\n",
    "    sharpe = m.get('sharpe', 0)\n",
    "    ret = m.get('expected_return', 0) * 100\n",
    "    vol = m.get('volatility', 0) * 100\n",
    "    dd = m.get('max_drawdown', 0) * 100\n",
    "    print(t)\n",
    "    print(f'  daily VaR 95%: {var95:.2f}%   daily CVaR 95%: {cvar95:.2f}%')\n",
    "    print(f'  Sharpe: {sharpe:.2f}   return: {ret:.1f}%   vol: {vol:.1f}%   avg max drawdown: {dd:.1f}%')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "paths = np.array(results['path_sample'])   # (n_sample, steps, n_assets): filtered sample, for display only\n",
    "n_assets = paths.shape[2]\n",
    "fig, axes = plt.subplots(1, n_assets, figsize=(5 * n_assets, 4), squeeze=False)\n",
    "\n",
    "for i, t in enumerate(used):\n",
    "    ax = axes[0, i]\n",
    "    for j in range(min(10, paths.shape[0])):\n",
    "        ax.plot(paths[j, :, i], alpha=0.3, linewidth=0.7)\n",
    "    ax.plot(paths[:, :, i].mean(axis=0), 'k--', linewidth=2, label='Mean of sample')\n",
    "    ax.set_title(t + ' sample paths')\n",
    "    ax.set_xlabel('Days')\n",
    "    ax.set_ylabel('Price')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "vr = results['variance_reduction']\n",
    "print('variance_reduction =', round(vr * 100, 2), '%')\n",
    "print('NOTE: this is max(0, 1 - var(simulated returns) / var(historical returns)).')\n",
    "print('It measures how far the simulation is from history, NOT PEMC variance reduction.')\n",
    "print('For real PEMC variance reduction see notebook 03.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "try:\n",
    "    from src.explainer.ollama_wrapper import LlamaExplainer\n",
    "    explainer = LlamaExplainer()\n",
    "    explanation = explainer.explain_simulation_results(\n",
    "        tickers=used,\n",
    "        expected_prices=expected_prices,\n",
    "        confidence_intervals=confidence_intervals,\n",
    "        risk_metrics=risk_metrics,\n",
    "        variance_reduction=vr,\n",
    "    )\n",
    "    print(explanation)\n",
    "except Exception as e:\n",
    "    print('[INFO] Explainer unavailable (is Ollama running?):', e)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def to_serializable(obj):\n",
    "    if isinstance(obj, np.integer):\n",
    "        return int(obj)\n",
    "    if isinstance(obj, np.floating):\n",
    "        return float(obj)\n",
    "    if isinstance(obj, np.ndarray):\n",
    "        return obj.tolist()\n",
    "    raise TypeError(str(type(obj)))\n",
    "\n",
    "os.makedirs('../data', exist_ok=True)\n",
    "with open('../data/results.json', 'w') as f:\n",
    "    json.dump(results, f, indent=2, default=to_serializable)\n",
    "print('[SUCCESS] Results saved to ../data/results.json')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}